# Rubrik adapter'ı — ölçeklenme eğrisinin ikinci noktası

Bu notebook adapter üretmiyor, **bir soruyu cevaplıyor**: tam koşu, kısa bir
koşunun verdiğinden ne kadar fazlasını veriyor?

Soru şuradan çıktı. Colab pilotu planlanan eğitimin **%1.75'ini** gördü
(~21 optimizer adımı, ~84 satır geçişi) ve şunu ölçtü: `present_score_mae`
0.813 → 0.631, `hallucinated_quotes` %3.25 → %1.82. Tam koşu 4800 satır
geçişi ve ölçülen 29 s/satır ile 38-47 saat — bir Kaggle oturumuna sığmıyor,
haftalık kotaya da sığmıyor. Sığdırma cambazlığına girişmeden önce cevabı
bilmek gereken soru, o 38 saatin satın aldığı şeyin ne olduğu.

**Pilotun sayıları buraya nokta olarak konamaz.** Pilot farklı bir eğitim
karışımıyla eğitildi ve kendi held-out'uyla (`rubric_eval_pilot.jsonl`)
ölçüldü; ana `rubric_eval.jsonl` onu görmüş satırlarla ölçerdi. O yüzden bu
koşu kendi tabanını **aynı oturumda** ölçüyor ve eğri üretim setinin kendi
eğrisi oluyor: 0 satır geçişi (taban) ve 800.

## Bütçe

| aşama | tahmin |
|---|---|
| kurulum + model indirme | ~20 dk |
| eğitim, 200 adım × 4 satır = 800 satır geçişi @ 29.1 s/satır | ~6.5 sa |
| ölçüm, taban + adapter, 60'ar satır | ~1.6 sa |
| **toplam** | **~8.4 sa** |

12 saatlik oturumda ~3.6 saat marj. Marj cömert çünkü bu koşu gözetimsiz:
29.1 s/satır %50 şaşsa bile eğitim biter ve ölçüm koşar.

`--grad-accum 4`, satırın 16'sı değil — pilotla aynı, ve aynı sebeple: aynı
satır geçişi için dört kat optimizer adımı, yani loss eğrisinde bakılacak bir
şey oluyor. Efektif batch 16 değil 4 olduğu için bu koşu tam koşunun bir
kısaltılmışı değil; ölçtüğü şey **veri miktarının getirisi**.

`--save-steps 25` gözetimsizliğin sigortası: oturum duvarına çarpsa bile son
checkpoint yazılmış olur.

In [ ]:
import glob, json, os, shutil, subprocess, sys
import torch

assert torch.cuda.is_available(), "GPU acik degil - Settings > Accelerator > GPU T4"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
print("bellek: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

# Fail here, in five seconds, rather than after an 8 GB download. A P100 is
# sm_60: Kaggle's torch build does not support it at all, and bitsandbytes needs
# sm_75 for 4-bit NF4. The Flutter run landed on one because kernel-metadata
# omitted machine_shape, and the error arrived half an hour in wearing a
# different mask.
assert cap >= (7, 5), (
    f"sm_{cap[0]}{cap[1]} yetersiz - 4-bit NF4 icin T4 (sm_75) gerekiyor. "
    "Settings > Accelerator > GPU T4 x2")

In [ ]:
# Qwen3 icin transformers >= 4.51 gerekiyor; Kaggle imaji eskiyse sessizce
# 'unknown architecture' ile duser.
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)
# torchao kaldiriliyor, yukseltilmiyor. peft'in LoRA dispatcher'i sardigi her
# kuantize OLMAYAN Linear icin is_torchao_available() soruyor ve o fonksiyon
# uyumsuz surumde False donmek yerine ImportError firlatiyor. Kaggle imaji
# 0.10.0 tasiyor, peft ('peft>=0.11' artik 0.20'ye cozuluyor) >0.16.0 istiyor.
#
# Tuzak fp16 kolunda: 4-bit'te bitsandbytes kendi Linear4bit'ini once
# eslestirdigi icin dispatcher'a hic varilmiyor. colab-pilot-eval bunu bir kez
# odedi ve cozdu; buraya tasinmadigi icin rubric-curve-eval ayni duvara carpti
# — taban olcumu bittikten sonra, adapter gecisinin ilk saniyesinde.
#
# Silmek find_spec'i None yapar ve kontrol False doner, ki dogru cevap odur:
# torchao nicemlemesi kullanmiyoruz. Yukseltmek torch'u da suruklerdi.
!pip -q uninstall -y torchao 2>&1 | tail -1


In [ ]:
def find_mount(slug, marker):
    """Locate one input mount by the dataset/kernel slug in its path.

    Not by filename. A kernel attached with kernel_sources contributes the whole
    of its /kaggle/working, which includes its own copies of the data files and
    the scripts — so searching for a data file finds two mounts and picks between
    them by luck. The slug is the only thing that distinguishes them.

    Recursive on top of that, because the mount depth is not a promise: the same
    dataset has appeared directly under /kaggle/input and, on the next run, one
    level deeper under /kaggle/input/datasets.
    """
    hits = [p for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
            if slug.split("/")[-1] in p]
    assert hits, (f"'{slug}' bagli degil (aranan: {marker}). "
                  f"Kaggle > Notebook > Add Input, ve surumun islenmesi bitmis olmali.")
    return os.path.dirname(sorted(hits, key=len)[0])


for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", sorted(files)[:4], "..." if len(files) > 4 else "")
    if root.count("/") > 6:
        dirs.clear()

In [ ]:
WORK = "/kaggle/working"
DATA = find_mount("emrahik/rubric-dataset", "rubric_train.jsonl")
print("veri seti:", DATA)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(DATA):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{DATA}/{f}", dst)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))
print(sorted(os.listdir(f"{WORK}/data")))

# The dataset carries the scripts, and this notebook passes --max-steps, which
# the pre-pilot train_qlora_qwen.py does not have. A stale dataset version fails
# here rather than six hours in, and the message says which half is behind.
helptext = subprocess.run([sys.executable, "train_qlora_qwen.py", "--help"],
                          capture_output=True, text=True).stdout
for flag in ("--max-steps", "--save-steps"):
    assert flag in helptext, (
        f"train_qlora_qwen.py '{flag}' bilmiyor — Kaggle dataset'i eski. "
        "peft/kaggle/push.sh calistir, surumun islenmesini bekle, sonra bu "
        "kernel'i yeniden push et.")
print("script guncel: --max-steps ve --save-steps var")

## 1. Eğitim — 800 satır geçişi

Tam koşunun altıda biri. `--max-steps` seçildi, veri setini küçültmek değil:
veri miktarı neyin öğrenildiğini de değiştirir, adım limiti ise yalnızca ne
kadar süre öğrenildiğini. Ölçmek istediğimiz şey ikincisi.

`--max-seq-len 2560` sabit ve sabit kalacak. `measure_tokens.py` ölçtü:
karışımın en uzun satırı 2477 token, p95 2308. Kırpma soldan olduğu için
daha düşük bir değer vakanın **başını** atar ve modele hiç görmediği kanıta
atıf yapmayı öğretir — gayet normal görünen bir loss'la. Bu koşu bir bütçe
koşusu, ama bütçe buradan çıkarılmıyor.

`PYTORCH_ALLOC_CONF=expandable_segments:True` boilerplate değil: rubric-train'in
ilk koşusu adım 0'da düştü, 14.56 GB'ın 2.54 GB'ı boşken 2.58 GB istendi —
Qwen3'ün 151.936'lık vocab'ının logits tensörü, ve 1.07 GB ayrılmış-ama-
kullanılmayan bellek. Fragmentasyon.

In [ ]:
MAX_STEPS = 200
GRAD_ACCUM = 4
OUT = "out/rubric-curve-800"     # 200 adim x 4 satir = 800 satir gecisi

env = dict(os.environ, PYTORCH_ALLOC_CONF="expandable_segments:True",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

# subprocess.run, `!` degil. `!`'in cikis kodu hicbir yere gitmez: rubric-train'in
# ilk kosusunda egitim adim 0'da CUDA OOM ile oldu, hucre devam etti, ve Kaggle
# kernel'i COMPLETE kaydetti — geriye kanit olarak yalnizca bos bir dizin kaldi.
r = subprocess.run([sys.executable, "train_qlora_qwen.py",
                    "--train", "data/rubric_train.jsonl",
                    "--eval", "data/rubric_eval.jsonl",
                    "--out-dir", OUT,
                    "--max-seq-len", "2560",
                    "--max-steps", str(MAX_STEPS),
                    "--grad-accum", str(GRAD_ACCUM),
                    "--batch-size", "1",
                    "--save-steps", "25"], env=env)
assert r.returncode == 0, f"egitim coktu (exit {r.returncode}) — log yukarida"

# Cikis kodu 0 yetmiyor: agirliklarin varligi ayrica kontrol edilmeli, yoksa
# Save Version bos bir dizini kaydeder ve olcum onu okur.
assert os.path.exists(f"{OUT}/adapter_model.safetensors"), \
    f"egitim bitti ama adapter yazilmamis — {OUT} bos"

## 2. Ölçüm — taban ve adapter, aynı oturumda

Taban aynı oturumda ölçülüyor çünkü bu koşunun ürettiği tek şey **fark**. Ayrı
oturumda alınmış bir taban rakamı farklı kütüphane sürümleriyle gelir ve
aradaki farkın ne kadarının adapter olduğu söylenemez.

60 satır, `rubric-eval`'in üretim ayarıyla aynı. Contrast yok: bu koşunun
sorusu "kuralı mı öğrendi bankayı mı" değil, "daha fazla eğitim ne veriyor".
Contrast tam koşunun ölçümüne ait ve ~40 dakika daha yerdi.

**Hangi sayıya bakılacak.** `absent_rate` değil — taban onu zaten %89 yapıyor
ve pilot da yerinde saydığını gösterdi (89.9 → 90.9). Bir metriğin tavanı
adapter'ın kazancını değil, tabanın yeterliliğini ölçer; Flutter v8 tam olarak
burada yanıldı. Karar verecek olanlar `present_score_mae` (düşük iyi) ve
`hallucinated_quotes`.

In [ ]:
r = subprocess.run([sys.executable, "rubric_eval.py",
                    "--data", "data/rubric_eval.jsonl",
                    "--base-model", "Qwen/Qwen3-4B-Instruct-2507",
                    "--adapter", OUT,
                    "--limit", "60",
                    "--out", "out/curve_800.json"])
assert r.returncode == 0, f"olcum coktu (exit {r.returncode}) — log yukarida"

In [ ]:
res = json.load(open("out/curve_800.json"))
print(json.dumps(res, indent=2, ensure_ascii=False))

b, a = res["base"], res["adapter"]
print("\n800 satir gecisi, uretim karisimi ve uretim eval seti\n")
print(f"{'olcum':<22}{'taban':>10}{'adapter':>10}")
for k in ("schema_valid", "completed", "absent_rate",
          "present_score_mae", "hallucinated_quotes"):
    bv, av = b.get(k), a.get(k)
    fmt = lambda v: "-" if v is None else f"{v:.3f}"
    print(f"{k:<22}{fmt(bv):>10}{fmt(av):>10}")

# Bir sonraki karar bu iki satirdan cikiyor. Kazanc pilotun gordugu mertebedeyse
# egri erken duzlesiyor demektir ve tam kosunun 38 saati gerekcesiz kalir.
print("\nKarar metrikleri: present_score_mae ve hallucinated_quotes.")
print("absent_rate tavanda — tabanin yeterliligini olcer, adapter'in kazancini degil.")

m = "out/rubric-curve-800/train_metrics.json"
if os.path.exists(m):
    print("\n" + json.dumps(json.load(open(m)), indent=2, ensure_ascii=False))